# Colab: prepare data and run a short training test

Select a Colab T4 GPU kernel and run all cells. The training cell prepares everything again after a runtime restart. This default smoke test is not a meaningful benchmark. The repository is cloned from GitHub, so local uncommitted changes are not transferred. Data and results in `/content` disappear when the server is removed.


In [ ]:
REPO_URL = "https://github.com/EdwinKam/emg2qwerty.git"
PROJECT_REF = "376923b"  # Published commit; use another published ref if needed.
MAX_STEPS = 10  # Increase for real training.
BATCH_SIZE = 4


In [ ]:
import os, subprocess, sys, time, tarfile
from pathlib import Path
try:
    import google.colab
    import torch
except ImportError as exc:
    raise RuntimeError('Select a hosted Colab GPU kernel first.') from exc
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
root = Path('/content')
repo = root / 'emg2qwerty'
data = root / 'emg2qwerty-data-2021-08'
archive = root / 'emg2qwerty-data-2021-08.tar.gz'
env = root / 'emg2qwerty-env'
mamba = root / 'bin/micromamba'
url = 'https://fb-ctrl-oss.s3.amazonaws.com/emg2qwerty/emg2qwerty-data-2021-08.tar.gz'
def run(args, **kwargs):
    print('+', ' '.join(map(str, args)), flush=True)
    subprocess.run([str(x) for x in args], check=True, **kwargs)
git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
if not repo.exists():
    run(['git', 'clone', REPO_URL, repo], env=git_env)
if not (repo / '.git').exists():
    raise RuntimeError(f'{repo} is not a Git checkout; inspect it before continuing.')
run(['git', '-C', repo, 'fetch', 'origin'], env=git_env)
if subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain', '--untracked-files=no'], text=True).strip():
    raise RuntimeError(f'{repo} has local changes; preserve them before continuing.')
run(['git', '-C', repo, 'checkout', '--detach', PROJECT_REF], env=git_env)
print('Code commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip(), flush=True)
if not mamba.exists():
    run(['bash', '-c', 'set -o pipefail; curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba'])
if not (env / 'bin/python').exists():
    run([mamba, 'create', '-y', '-p', env, '-c', 'conda-forge', 'python=3.10', 'pip=24.0'])
python = env / 'bin/python'
marker = env / '.dependencies-ready'
if not marker.exists():
    run([python, '-m', 'pip', 'install', '-r', repo / 'requirements.txt'])
    run([python, '-m', 'pip', 'install', '--no-deps', '-e', repo])
    run([python, '-c', 'import torch, torchaudio, pytorch_lightning, hydra, kenlm; assert torch.cuda.is_available(), "PyTorch cannot access GPU"'])
    marker.touch()
check = """import sys, yaml
from pathlib import Path
cfg = yaml.safe_load(Path(sys.argv[1]).read_text())
root = Path(sys.argv[2])
missing = [root / (s['session'] + '.hdf5') for split in ('train', 'val', 'test') for s in cfg['dataset'][split] if not (root / (s['session'] + '.hdf5')).is_file()]
if missing:
 print('Missing', len(missing), 'configured data files', flush=True)
 sys.exit(1)
"""
def data_ready():
    return subprocess.run([str(python), '-c', check, str(repo / 'config/user/single_user.yaml'), str(data)]).returncode == 0
if not data_ready():
    if not archive.exists():
        partial = Path(str(archive) + '.part')
        run(['curl', '-fL', '--retry', '3', '-o', partial, url])
        partial.replace(archive)
    with tarfile.open(archive, 'r:gz') as tf:
        for member in tf.getmembers():
            target = (root / member.name).resolve()
            if not target.is_relative_to(root.resolve()) or member.issym() or member.islnk():
                raise RuntimeError(f'Unsafe archive entry: {member.name}')
        tf.extractall(root)
    if not data_ready():
        raise RuntimeError('Extracted dataset is missing required sessions; inspect /content.')
link = repo / 'data'
if link.is_symlink():
    if link.resolve() != data.resolve():
        raise RuntimeError(f'{link} points elsewhere; inspect it before continuing.')
elif link.exists():
    raise RuntimeError(f'{link} already exists; inspect it before continuing.')
else:
    link.symlink_to(data, target_is_directory=True)
run_dir = root / 'emg2qwerty-results' / time.strftime('%Y%m%d-%H%M%S')
run_dir.mkdir(parents=True, exist_ok=False)
command = [python, '-m', 'emg2qwerty.train', 'user=single_user', 'trainer.accelerator=gpu', 'trainer.devices=1', 'trainer.max_epochs=1', f'trainer.max_steps={MAX_STEPS}', f'batch_size={BATCH_SIZE}', 'num_workers=2', 'trainer.limit_val_batches=2', 'trainer.limit_test_batches=1', f'hydra.run.dir={run_dir}']
print('Training; results:', run_dir, flush=True)
with (run_dir / 'console.log').open('w') as log:
    process = subprocess.Popen([str(x) for x in command], cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    if process.wait() != 0:
        raise RuntimeError(f'Training failed; see {run_dir / "console.log"}')
import shutil
result_zip = shutil.make_archive(str(run_dir), 'zip', run_dir)
print('Download the results archive:', result_zip, 'before removing the server.', flush=True)


## Save results and stop the GPU

Download the ZIP under `/content/emg2qwerty-results` using the Colab Contents view, or copy it to Google Drive. Verify the results are saved, then select **Colab → Remove Server**. A fresh runtime downloads and extracts the dataset again.
